# Agent Evaluation With MLflow

This notebook demonstrates how to use MLflow's evaluate, scorer, and LLM-judge features to evaluate an agent's performance before deployment.

## Prerequisites

Complete the [Quick Start](../readme.md#quick-start) setup before running this notebook.

You will also need:
- Access to an OpenAI-compatible LLM endpoint (e.g. OpenAI, vLLM, llama.cpp, Ollama, etc.)
- [Sign up for a National Park Service API Key](https://www.nps.gov/subjects/developer/get-started.htm) - it is completely free and quick
  - After you receive your email, add the NPS API Key to your environment below
- Choose one of the MLflow configurations in the cell below

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# Load environment variables from .env file (searches current and parent directories)
load_dotenv(find_dotenv())

## Local MLFlow - Remember to start the server first! `uv run mlflow server --port 5001`
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5001"
os.environ["MLFLOW_TRACKING_AUTH"] = ""
os.environ["MLFLOW_EXPERIMENT_NAME"] = "nps-agent"

## RHOAI - Remember to log in to your project first!
# os.environ["MLFLOW_TRACKING_URI"] = "https://<your-cluster>/mlflow"
# os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes"
# os.environ["MLFLOW_EXPERIMENT_NAME"] = "nps-agent"

## E.g. For Ollama, uncomment and set these:
# os.environ["OPENAI_API_KEY"] = "EMPTY"
# os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"
# os.environ["OPENAI_MODEL_NAME"] = "gpt-oss:20b"

# Check that required vars are set, raise an error if not
required_vars = ["MLFLOW_TRACKING_URI", "OPENAI_API_KEY", "NPS_API_KEY"]
missing_vars = [var for var in required_vars if not os.getenv(var)]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

## Set Up Autologging

As [shown in the previous notebook](./1_trace.ipynb#instrumenting-agents-with-autologgers), instrumenting tracing for agents in MLflow is very simple.

We'll use the OpenAI Agents SDK in this example, so we set up autologging using the OpenAI autologger.

In [ ]:
import mlflow

# Now enable MLflow tracing - Just one line!
mlflow.openai.autolog()

## Create an NPS Agent

We will now create an agent that can answer questions about the U.S. National Park Service (NPS).

This agent will have access to an MCP server with five tools:

- `search_parks`: Search for national parks by state, park code, or query
- `get_park_alerts`: Get current alerts for a specific park
- `get_park_campgrounds`: Get campground information for a park
- `get_park_events`: Get upcoming events for a park
- `get_visitor_centers`: Get visitor center information for a park

You can view the MCP server code [here](./nps_mcp_server.py). 
The MCP server is run using STDIO transport to show a proper development environment configuration.

In [ ]:
from agents import Agent, Runner, set_default_openai_client
from agents.mcp import MCPServerStdio
from openai import AsyncClient

AGENT_INSTRUCTIONS = (
    "You are a helpful National Parks Service assistant. "
    "Use the available tools to answer questions about national parks, "
    "events, activities, campgrounds, and visitor information. "
)

# Create a function to run the agent
async def run_nps_agent(prompt) -> str:
    """Run the NPS agent with MCP tools and return the text response."""
    command = "uv"
    args = ["run", "fastmcp", "run", "./nps_mcp_server.py"]
    env = {**os.environ, "NPS_API_KEY": os.environ.get("NPS_API_KEY", "")}
    async with MCPServerStdio(params={"command": command, "args": args, "env": env}) as mcp_server:
        # Configure OpenAI-compatible endpoint
        async_client = AsyncClient(
            base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
            api_key=os.environ.get("OPENAI_API_KEY", ""),
        )
        set_default_openai_client(client=async_client)

        # Create the agent
        agent = Agent(
            name="NPS Agent",
            instructions=AGENT_INSTRUCTIONS,
            mcp_servers=[mcp_server],
            model=os.environ.get("OPENAI_MODEL_NAME", "gpt-4o"),
        )

        # Run the agent
        result = await Runner.run(agent, prompt)
        return result.final_output

In [ ]:
from IPython.display import display, Markdown

prompt = "What campgrounds are available at the grand canyon?"
res = await run_nps_agent(prompt=prompt)
display(Markdown(res))

When working in Jupyter notebooks with a local MLflow server, you'll see the trace appear directly below any cell that uses a traced function.
This gives you a quick way to view the whole trace from your development environment.

To view this same trace in the MLflow UI, navigate to Experiments > Default > Traces. 
Here you can see all traces that come into your experiment.

<center>
<img src="./images/nps_agent_trace.png" alt="drawing" width="75%"/>
<br>
<em>NPS Agent Trace</em>
</center>

Now that we have an agent created, we can start evaluating it.

## Agent Evaluation

Agent evaluation comes in two phases. 
First you have the "inner loop", which is focused on evaluating the performance of an agent during development.
Then you have the "outer loop", which is focused on evaluating the performance of an agent in a production environment.

<center>
<img src="./images/inner_outer_loop.png" alt="drawing" width="75%"/>
<br>
<em>Diagram of the Inner and Outer Loops</em>
</center>

In this section we will focus on the inner loop.
The outer loop will be covered in the [Observe section](../3_observe/) of this repo.

There are four main stages in inner loop evaluations:

1. Gathering test cases of "golden" inputs and expectation pairs into a dataset
2. Building scorers to grade the agent and measure its performance
3. Running the test cases against the agent and using the scorers to evaluate its output
4. Having human experts review the evaluation results and provide their own feedback

We will cover each of these stages in the following sections.

## 1. Gathering Test Cases

The first step in inner loop evaluations is to build a dataset of "golden" inputs and expectation pairs.

You can create and save these datasets to MLflow using the `mlflow.genai.datasets` package.

For each test case you'll need to set two keys: `inputs` and `expectations`.

`inputs` is a dictionary containing all the inputs to your prediction function. 
It should have the same keys as the arguments in your prediction function, and each will be passed to the prediction function during evaluation.

`expectations` is a dictionary containing all the expectations you have for your agent response.
This dictionary will be passed to scorers along with the agent traces during evaluation.
You'll put the information that the scorers need to be able to assess the traces here.
There are some special keys to be aware of:

- `expected_response`: This will be treated as the expected final response from the agent for scorers such as `Correctness` and `Equivalence`. This should be a string.
- `expected_tool_calls`: This will be treated as the list of expected tool calls for scorers such as `ToolCallCorrectness`. This should be a list of dictionaries with either {"name"} or {"name", "arguments"} keys.
- `expected_facts`: Mainly for RAG apps, this will be treated as the required facts that must be retrieved by the retriever. This should be a list of strings.

We will use the `expected_response` key in this demonstration.

In [ ]:
from mlflow.genai.datasets import create_dataset

# Start by creating a new empty dataset on your server
dataset_name = "nps_agent_dataset"
dataset = create_dataset(
    name=dataset_name,
    tags={"stage": "validation", "version": "1"},
)

# Write up your test cases
test_cases = [
    {
        "inputs": {"prompt": "What time does Acadia's main visitor center open during the summer?"},
        "expectations": {
            "expected_response": "8:30 AM",
        }
    },
    {
        "inputs": {"prompt": "What park has the code KLGO?"},
        "expectations": {
            "expected_response": "Klondike Gold Rush National Historical Park",
        }
    },
    {
        "inputs": {"prompt": "How many campsites are at the Desert View Campground at the Grand Canyon?"},
        "expectations": {
            "expected_response": "49",
        }
    },
]

# Add the test cases to the dataset
dataset = dataset.merge_records(test_cases)
print(f"New dataset created: {dataset.dataset_id}")

You can visit the MLflow UI to view the dataset under Experiments > Default > Datasets


<center>
<img src="./images/dataset_view.png" alt="drawing" width="75%"/>
<br>
<em>Dataset view</em>
</center>

## 2. Build Scorers

Next we configure our judges/scorers for our agent.

MLflow comes with a [number of prepackaged LLM-judges](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/predefined/) that can be used for scoring.

We can define custom judges with our own criteria using the `make_judge` function.
Here we create a length judge to measure the length of the agent's responses.

We can also define code based scorers using the `@scorer` decorator.
Code based scorers can be used to execute deterministic code on the agent outputs, expectations, and/or traces.
Here we create a code based scorer to check if the request was replied to in an adequate amount of time (15 seconds).

In [ ]:
from typing import Literal

from mlflow.genai.scorers import Correctness, RelevanceToQuery, scorer
from mlflow.genai.judges import make_judge
from mlflow.entities import Trace, Feedback

# Set model
# NOTE: All LLM-judges use LiteLLM, which requires the model name in this format
#       For the OpenAI compatible endpoints, you also need the OPENAI_BASE_URL and OPENAI_API_KEY environment variables set for this to work
model = f'openai:/{os.getenv("OPENAI_MODEL_NAME")}' if os.getenv("OPENAI_MODEL_NAME") else "openai:/gpt-4o"

# Create a custom judge
length_judge = make_judge(
    name="Length",
    model=model,
    instructions=(
        "Determine if the response in the output object below is short, medium, or long.\n\n"
        "## Output Object\n\n{{ outputs }}\n\n"
        "Respond 'short' if the response is 1-3 sentences in length, 'medium' if it is 1-2 paragraphs, and 'long' if it is more than 2 paragraphs."
    ),
    feedback_value_type=Literal["short", "medium", "long"],
)

# Create a custom code-based scorer for traces
# This checks if the latency is within the maximum allowed latency of 15 seconds
@scorer(name="Latency")
def latency_check(trace: Trace):
    # Find the span for the agent run - AgentRunner.run in the case of OpenAI Agents SDK
    span = trace.search_spans(name="AgentRunner.run")[0]
    latency = (span.end_time_ns - span.start_time_ns) / 1e9
    max_allowed_latency = 15.0
    if latency > max_allowed_latency:
        return Feedback(value="no", rationale=f"LLM response time {latency:.2f}s exceeds the {max_allowed_latency}s limit.")
    else:
        return Feedback(value="yes", rationale=f"LLM response time {latency:.2f}s is within the {max_allowed_latency}s limit.")


# Collect scorers
scorers = [
    Correctness(model=model),           # Checks response correctness against expected_response
    RelevanceToQuery(model=model),      # Checks response is fully relevant to the original query
    length_judge,                       # Labels the response with short, medium, or long
    latency_check,                      # Checks that the response latency is within the maximum allowed latency
]

## 3. Run the Test Cases and Evaluate the Results

Now we can bring it all together under the `evaluate` function.

The evaluate function will take our test cases `dataset` object, send each of the inputs to our `run_nps_agent` function, then score the traces and outputs with the `scorers` we defined.

The results of the evaluation and all the traces will be saved to MLflow.
After running the below script, open MLflow in your browser and navigate to Experiments > Default > Evaluation Runs, then click on the latest run to view the results.

In [ ]:
from mlflow.genai import evaluate

# Run the evaluation job
results = evaluate(
    data=dataset,
    scorers=scorers,
    predict_fn=run_nps_agent,
)

# Print the results
print(results.metrics)

Your evaluation results should look something like this once completed.

<center>
<img src="./images/evaluation_results.png" alt="drawing" width="75%"/>
<br>
<em>Evaluation Results</em>
</center>

Looking at these results we can tell that out agent is doing well at answering questions correctly according to our ground truths (100% Correctness).
It's also really good at providing answers that are fully relevant to the query (100% Relevant).
However, it also looks like it's taking a bit longer than expected to respond (33% latency), so we may want to look at improving the speed of some of our tools or shortening our generation lengths to get folks their responses quicker.
We can also see that the response lengths vary, which we could use to perform an follow up analysis to see what types of questions get longer responses than others.

## 4. Human Feedback

Now that you have completed some automated evaluations, you can share these with your subject matter experts to review and add their own feedback.

If running on a remote tracking server, you can share the link to your evaluation run for them to review and add their feedback.

They can add new feedback or update existing feedback, including the automatic assessments.


### Add New Feedback

<center>
<img src="./images/add_feedback.png" alt="drawing" width="75%"/>
<br>
<em>Add feedback</em>
</center>

1. Navigate to your experiment
1. Click "Add Assessment" button
1. Select "Feedback" from the Assessment Type dropdown
1. Enter a descriptive name
1. Choose the appropriate data type
1. Enter your evaluation value
1. Add rationale explaining your evaluation reasoning
1. Click "Create" to record your feedback


### Update Existing Feedback

<center>
<img src="./images/edit_feedback.png" alt="drawing" width="75%"/>
<br>
<em>Edit feedback</em>
</center>

1. Locate the feedback entry you want to modify
1. Click the hamburger menu (⋮) next to the feedback entry
1. Select "Edit" from the dropdown menu
1. Modify the value, rationale, or other fields
1. Click "Save" to update the feedback